# ECML Paper Pipeline – Product A

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ---- Project imports ----
from src.data_loading.build_product_bundle import load_product_bundle
from src.common.alignment import align_views_and_target

from src.structural.run import run_structural_analysis

from src.single_view_mglvq.run import run_single_view_mglvq
from src.single_view_mglvq.mglvq_fast import MGLVQ

from src.m3glvq.normalize import build_normalized_view_list
from src.m3glvq.M3GLVQ import M3GLVQ
from src.m3glvq.analysis import run_m3glvq_cv_analysis
from src.m3glvq.run import (
    build_K_grid,
    run_m3glvq_persisted,
    select_top_runs,
    cluster_weight_profiles,
    summarize_clusters,
    extract_core_k_points,
    summarize_core_points,
    run_m3glvq_fixed_weights_persisted,
    summarize_fixed_profiles,
)

from src.analysis.run import run_pointwise_analysis, run_performance_analysis
from src.analysis.figures import load_joined_long_csv, plot_structure_vs_prediction_boxplot
from src.analysis.paper_tables import build_distance_distribution_table
from src.analysis.load_outputs import load_m3_fixed_oof_from_pickles

from src.analysis.paper_tables import (
    build_structural_characteristics_table,
    build_pearson_correlation_table,
    build_knn_overlap_table,
    build_agreement_table,
    build_single_view_baseline_table,
    build_structure_prediction_dominance_table,
    build_fixed_profile_dominance_table,
)



print("CWD:", Path.cwd())
print("REPO_ROOT:", REPO_ROOT)

## 1. Data Loading and Alignment

In [ ]:
# ----------------------------
# Paths and product setup
# ----------------------------
PRODUCT_CODE = "A"


Y_BASE = REPO_ROOT / "data" / "raw" / "y"
D_BASE = REPO_ROOT / "data" / "raw" / "D"


OUTPUT_ROOT = REPO_ROOT / "outputs"
STRUCTURAL_OUT = OUTPUT_ROOT / "structural"
SINGLE_VIEW_OUT = OUTPUT_ROOT / "single_view_mglvq"
M3_OUT = OUTPUT_ROOT / "m3glvq"
ANALYSIS_OUT = OUTPUT_ROOT / "analysis"

RUN_NAME = f"ph2_{PRODUCT_CODE}"

# ----------------------------
# Load bundle
# ----------------------------
bundle = load_product_bundle(
    product_code=PRODUCT_CODE,
    y_base_path=Y_BASE,
    d_base_path=D_BASE,
)

# ----------------------------
# Align views and target
# ----------------------------
aligned = align_views_and_target(
    matrices=bundle.matrices,
    y=bundle.y,
    order="y",
)

# ----------------------------
# Build normalized multi-view input
# ----------------------------
DL, D_views_norm = build_normalized_view_list(
    aligned.matrices,
    view_order=("NAICS", "HS", "AM"),
    q_low=5,
    q_high=95,
)

y_M3GLVQ = aligned.y
meta_df = pd.DataFrame({
    "CustomerCode": aligned.used_ids.astype(str)
})

print("Aligned sample size:", len(aligned.used_ids))
print("Views:", list(aligned.matrices.keys()))
print("DL shapes:", [x.shape for x in DL])


## 2. Block 1 – Structural Analysis

In [ ]:
structural_res = run_structural_analysis(
    matrices=bundle.matrices,
    y=bundle.y,
    run_name=RUN_NAME,
    out_root=STRUCTURAL_OUT,
    order="y",
    k=10,
    n_perm=50,
    seed=0,
    corr_methods=("pearson", "spearman"),
    knn_k=10,
    agree_quantile=0.10,
)

display(structural_res["single_summary_df"])

## 3. Block 2 – Single-View MGLVQ

In [ ]:
mglvq_res = run_single_view_mglvq(
    matrices=bundle.matrices,
    y=bundle.y,
    model_cls=MGLVQ,
    run_name=RUN_NAME,
    out_root=SINGLE_VIEW_OUT,
    order="y",
    do_grid=True,
    param_grid={
        "K": [3, 4, 5, 6, 7, 8, 9, 10],
        "T": [150],
        "track_path": [True],
    },
    n_splits=5,
    shuffle=True,
    random_state=42,
    pointwise_structure_df=structural_res["pointwise_structure_df"],
    show_progress=True,
    resume=True,
)

display(mglvq_res["summary_df"])

## 4. Block 3 – Multi-View M3GLVQ

### 4.1 Search

In [ ]:
K_grid = build_K_grid(
    K0_values=[3, 4, 5, 6, 7, 8, 9, 10],
    K1_values=[3, 4, 5, 6, 7, 8, 9, 10],
)

all_runs_fold_df, all_results = run_m3glvq_persisted(
    D=DL,
    y=y_M3GLVQ,
    model_cls=M3GLVQ,
    meta_df=meta_df,
    run_m3glvq_cv_analysis=run_m3glvq_cv_analysis,
    etas=[0.003, 0.0075, 0.01, 0.015, 0.025, 0.03, 0.04, 0.05],
    K_grid=K_grid,
    out_dir=M3_OUT / RUN_NAME / "search_runs",
    resume=True,
    save_full_result=True,
    verbose=True,
)

display(all_runs_fold_df.head())

### 4.2 Top-run clustering

In [ ]:
df_high, threshold = select_top_runs(
    all_runs_fold_df,
    metric_col="balanced_accuracy",
    quantile=0.9,
)

cluster_res = cluster_weight_profiles(
    df_high,
    v_cols=("vweight_0", "vweight_1", "vweight_2"),
    n_clusters=5,
    random_state=42,
    top_frac=0.20,
)

df_high_clustered = cluster_res["df_high"]

cluster_info = summarize_clusters(
    df_high_clustered,
    filtered_centers=cluster_res["filtered_centers"],
)

df_high_core = extract_core_k_points(
    df_high_clustered,
    X=cluster_res["X"],
    labels=cluster_res["labels"],
    filtered_centers=cluster_res["filtered_centers"],
    n_clusters=5,
    n_core=20,
)

cluster_core_df = summarize_core_points(df_high_core)
cluster_core_df["core_center"] = cluster_core_df["core_center"].apply(lambda x: np.round(x, 3))

display(cluster_core_df)

### 4.3 Cluster plot

In [ ]:
df_plot = cluster_res["df_high"].copy()
core_mask = cluster_res["core_mask"]
centers_pca = cluster_res["centers_pca"]

plt.figure(figsize=(8, 6))

plt.scatter(
    df_plot.loc[~core_mask, "PC1"],
    df_plot.loc[~core_mask, "PC2"],
    color="lightgray",
    alpha=0.35,
    s=30,
    label="remaining points"
)

clusters = sorted(df_plot["cluster"].unique())
for c in clusters:
    mask_c_core = (df_plot["cluster"] == c) & core_mask
    plt.scatter(
        df_plot.loc[mask_c_core, "PC1"],
        df_plot.loc[mask_c_core, "PC2"],
        s=55,
        alpha=0.9,
        label=f"Cluster {c} core"
    )

plt.scatter(
    centers_pca[:, 0],
    centers_pca[:, 1],
    marker="X",
    s=220,
    edgecolor="black",
    facecolor="none",
    linewidth=1.8,
    label="filtered centers"
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of top-10% M3GLVQ runs with core-based cluster centers")
plt.legend()
plt.tight_layout()
plt.show()

### 4.4 Prototype-count recommendations per cluster

In [ ]:
clusters_keep = [1, 4]

df_core_sel = df_high_core[
    (df_high_core["core20"]) & (df_high_core["cluster"].isin(clusters_keep))
].copy()

k_combo_stats = (
    df_core_sel
    .groupby(["cluster", "K_0", "K_1"], as_index=False)
    .agg(
        count=("cluster", "size"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        std_balanced_accuracy=("balanced_accuracy", "std"),
        mean_recall=("recall", "mean"),
        std_recall=("recall", "std"),
    )
    .sort_values(
        ["cluster", "count", "mean_balanced_accuracy", "K_0", "K_1"],
        ascending=[True, False, False, True, True]
    )
)

top_n = 10
for c in clusters_keep:
    print(f"\n===== Cluster {c} =====")
    display(
        k_combo_stats[k_combo_stats["cluster"] == c]
        .head(top_n)
        .reset_index(drop=True)
    )

### 4.5 Fixed profiles

In [ ]:
fixed_profiles = [
    {"profile_name": "C0_primary",   "weights": [0.546, 0.280, 0.175], "K": {0: 6, 1: 5}},
    {"profile_name": "C0_secondary", "weights": [0.546, 0.280, 0.175], "K": {0: 4, 1: 6}},
    {"profile_name": "C3_primary",   "weights": [0.376, 0.323, 0.301], "K": {0: 4, 1: 3}},
    {"profile_name": "C3_secondary", "weights": [0.376, 0.323, 0.301], "K": {0: 6, 1: 3}},
    {"profile_name": "C4_primary",   "weights": [0.815, 0.181, 0.004], "K": {0: 3, 1: 5}},
    {"profile_name": "C4_secondary", "weights": [0.815, 0.181, 0.004], "K": {0: 4, 1: 4}},
]

all_fixed_fold_df, all_fixed_results = run_m3glvq_fixed_weights_persisted(
    D=DL,
    y=y_M3GLVQ,
    model_cls=M3GLVQ,
    meta_df=meta_df,
    run_m3glvq_cv_analysis=run_m3glvq_cv_analysis,
    profiles=fixed_profiles,
    T=150,
    n_splits=10,
    random_state=42,
    out_dir=M3_OUT / RUN_NAME / "fixed_runs",
    resume=True,
    save_full_result=True,
    verbose=True,
)

summary_fixed = summarize_fixed_profiles(all_fixed_fold_df)
display(summary_fixed.round(4))

### 4.6 Save block-3 summary tables

In [ ]:
out_m3 = M3_OUT / RUN_NAME
out_m3.mkdir(parents=True, exist_ok=True)

cluster_core_df.to_csv(out_m3 / "cluster_core_df.csv", index=False)
summary_fixed.to_csv(out_m3 / "summary_fixed.csv", index=False)

## 5. Integrated Paper Tables

### 5.1 Distance Distribution Table

In [ ]:
distance_table = build_distance_distribution_table(
    D_views_norm=D_views_norm,
    label_map={
        "NAICS": "Industry (NAICS)",
        "HS": "Products (HS)",
        "AM": "Applications (AP)",
    },
)

display(distance_table.round(3))

### 5.2 Structural characteristics of the individual dissimilarity spaces.

In [ ]:
single_summary_df = pd.read_csv(
    STRUCTURAL_OUT / RUN_NAME / "single_view_summary.csv"
)

structural_table = build_structural_characteristics_table(single_summary_df)
display(structural_table.round(4))

### 5.3 Pearson correlation between vectorized pairwise dissimilarities of the three

In [ ]:
structural_run_dir = STRUCTURAL_OUT / RUN_NAME
table3 = build_pearson_correlation_table(structural_run_dir)
display(table3.round(3))
table3.round(6).to_csv(ANALYSIS_OUT / RUN_NAME / "summaries" / "paper_table_3_correlation.csv")

### 5.4 Mean overlap of kNN neighborhoods between dissimilarity views for kNN = 10.

In [ ]:
table4 = build_knn_overlap_table(structural_run_dir, k=10)
display(table4.round(3))
table4.round(6).to_csv(ANALYSIS_OUT / RUN_NAME / "summaries" / "paper_table_4_knn_overlap.csv")

### 5.5 Pairwise agreement of similarity relations between dissimilarity views based on a 10% quantile threshold.

In [ ]:
table5 = build_agreement_table(structural_run_dir, quantile=0.10, subset="all")
display(table5.round(3))
table5.round(6).to_csv(ANALYSIS_OUT / RUN_NAME / "summaries" / "paper_table_5_agreement.csv", index=False)

### 5.6 Performance of the single-view MGLVQ baselines based on out-of-fold predictions under the selected hyperparameter configuration.

In [ ]:
single_view_summary_df = pd.read_csv(
    SINGLE_VIEW_OUT / RUN_NAME / "summary.csv"
)

table6 = build_single_view_baseline_table(
    single_view_summary_df,
    label_map={
        "naics": "Industry (NAICS)",
        "hs": "Products (HS)",
        "am": "Applications (AP)",
    },
)

display(table6.round(3))

table6.to_csv(
    ANALYSIS_OUT / RUN_NAME / "summaries" / "paper_table_6_single_view_baselines.csv",
    index=False,
)

### 5.7 Number of customers grouped by structural dominance 
(columns, defined by the view with the highest local kNN purity) and prediction dominance (rows, defined by which of the three single-view MGLVQ models predict the customer correctly).

In [ ]:
wide_df = pd.read_csv(
    Path.cwd().parent / "outputs" / "analysis" / "ph2_A" / "pointwise" / "single_view_pointwise_comparison_wide.csv"
)

table7 = build_structure_prediction_dominance_table(
    wide_df,
    structure_metric="knn_purity",
    label_map={
        "naics": "Industry (NAICS)",
        "hs": "Products (HS)",
        "am": "Applications (AP)",
    },
)

display(table7)

table7.to_csv(
    Path.cwd().parent / "outputs" / "analysis" / "ph2_A" / "summaries" / "paper_table_7_structure_prediction_dominance.csv",
    index=False,
)

### 5.8 Number of correctly classified customers for the selected fixed-weight M3GLVQ variants across the prediction-dominance groups obtained from the singleview MGLVQ analysis. 
The groups indicate whether a customer was correctly classified only by the Industry, Products, or Applications single-view model, by more than one but not all models (Mixed), by all models (All correct), or by none of them (All wrong). For each M3GLVQ profile, the value in parentheses denotes the difference relative to the total number of customers in the respective group.

In [ ]:
analysis_out = Path.cwd().parent / "outputs" / "analysis" / "ph2_A"
m3_fixed_dir = Path.cwd().parent / "outputs" / "m3glvq" / "ph2_A" / "fixed_runs"

wide_df = pd.read_csv(
    analysis_out / "pointwise" / "single_view_pointwise_comparison_wide.csv"
)

fixed_oof_df = load_m3_fixed_oof_from_pickles(m3_fixed_dir)

table9 = build_fixed_profile_dominance_table(
    wide_df=wide_df,
    fixed_oof_df=fixed_oof_df,
    profile_map={
        "Cluster 0": "C0_primary",
        "Cluster 3": "C3_primary",
        "Cluster 4": "C4_primary",
    },
    label_map={
        "naics": "Industry (NAICS)",
        "hs": "Products (HS)",
        "am": "Applications (AP)",
    },
)

display(table9)

table9.to_csv(
    analysis_out / "summaries" / "paper_table_9_fixed_profile_by_dominance_group.csv",
    index=False,
)

## 6. Paper Figure: Structure vs Prediction

In [ ]:
joined_path = ANALYSIS_OUT / RUN_NAME / "pointwise" / "structure_single_view_joined_long.csv"
joined_df = load_joined_long_csv(joined_path)

fig = plot_structure_vs_prediction_boxplot(
    joined_df=joined_df,
    metric="knn_purity",
    title="Local Neighborhood Purity (kNN)",
    out_path=ANALYSIS_OUT / RUN_NAME / "figures" / "ecml_boxplot_knn_only.png",
    matrix_order=["naics", "hs", "am"],
    label_map={
        "naics": "Industry",
        "hs": "Products",
        "am": "Applications",
    },
)

plt.show()